In [1]:
import csv
import pandas as pd
import numpy as np
from collections import defaultdict

In [2]:
base = "CME"

In [3]:
if base == "CME":
    sec = "Dino"
if base == "Dino":
    sec = "CME"

df_comp = pd.read_csv(f"Base Output/Comparison.csv")

M_comp = df_comp.to_numpy()
display(df_comp)

,ID (CME),ID (Dino),x (CME),y (CME),z (CME),x (Dino),y (Dino),z (Dino),t,t_start (CME),...,FI (CME),FI (Dino),Track Length (CME),Track Length (Dino),Feature 0,Feature 1,Feature 2,Distance,Multi ID (Dino),Multi Distance (Dino)
0,2.0,237.0,291.30,226.51,21.240,289.95996,225.44984,21.011475,1.0,1.0,...,229.960,77.44,99.0,99.0,-5.560,-5.60500,-0.8926,1.723911,237,1.72
1,2.0,237.0,291.98,226.98,20.920,290.60168,225.96657,20.809637,2.0,1.0,...,242.520,80.75,99.0,99.0,-7.723,-1.64500,-0.6924,1.714347,237,1.71
2,2.0,237.0,291.70,226.58,20.900,290.33856,225.55232,20.688269,3.0,1.0,...,252.340,84.94,99.0,99.0,-6.344,-1.44500,-1.2480,1.718859,237,1.72
3,2.0,237.0,291.69,227.07,20.840,290.68555,226.09846,20.659414,4.0,1.0,...,272.540,85.70,99.0,99.0,-6.938,0.06042,-1.7070,1.409050,237,1.41
4,2.0,237.0,292.56,227.12,20.730,291.28845,226.16327,20.675010,5.0,1.0,...,246.290,77.94,99.0,99.0,-6.254,-0.46780,-1.9950,1.592230,237,1.59
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
124431,10789.0,11435.0,528.42,544.83,154.673,528.88763,543.35535,149.208000,99.0,98.0,...,119.250,29.19,2.0,17.0,16.730,-9.25000,0.9000,5.679744,NaN,NaN
124432,10790.0,12050.0,563.88,119.02,155.781,562.20996,123.98951,145.984480,98.0,98.0,...,32.322,26.78,2.0,26.0,4.040,3.87900,-1.8310,11.111115,NaN,NaN
124433,10790.0,12325.0,563.70,119.53,154.632,557.89760,110.49770,164.956360,99.0,98.0,...,34.478,34.40,2.0,29.0,-9.990,10.34000,2.2950,14.894385,NaN,NaN
124434,10791.0,14310.0,523.25,139.19,153.654,526.07980,139.12753,165.447710,98.0,98.0,...,29.372,31.05,2.0,14.0,-1.944,20.64000,4.1900,12.128614,NaN,NaN


In [4]:
ID_vec = defaultdict(list)

for ID in M_comp:
    ID_vec[ID[0]].append(ID)

for ID_val in ID_vec:
    ID_vec[ID_val] = np.array(ID_vec[ID_val]) 

ID_list = np.zeros((len(ID_vec),5), dtype=object)

In [5]:
for i, ID in enumerate(ID_vec):
    ID_val = ID_vec[ID]
    track_coverage = np.sum(ID_val[:,18] < 3.5)
    missing_t = ID_val[ID_val[:, 18] > 3.5][:,8]
    ID_list[i][0] = ID_val[0][0]
    ID_list[i][1] = ID_val[0][13]
    ID_list[i][2] = track_coverage
    ID_list[i][3] = missing_t.tolist()
    block_starts = np.where(np.diff(ID_val[:,1]) != 0)[0] + 1
    block_starts = np.insert(block_starts, 0, 0)
    track_index = np.where(np.diff(ID_val[:,1]) != 0)[0] + 1
    track_index = np.insert(track_index, 0, 0)
    y_ID = ID_val[:,1][block_starts]
    track_lengths = ID_val[:,14][track_index]
    y = np.stack((y_ID,block_starts+1,track_lengths), axis=1)
    y_all = ';'.join([','.join(map(str, row)) for row in y])
    ID_list[i][4] = y_all

In [6]:
df_ID = pd.DataFrame(ID_list, columns = [f"ID ({base})",f"Track Length ({base})",f"Track Coverage","Missing Time Points",f"{sec} Data"])

In [7]:
df_ID.to_csv(f"Base Output/ID_Coverage.csv", index=False)